In [ ]:
import requests

url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
response = requests.get(url)

with open("names.txt", "wb") as f:
    f.write(response.content)

print("names.txt downloaded successfully.")

names.txt downloaded successfully.


In [ ]:
words = open("names.txt", 'r').read().splitlines()

In [ ]:
len(words)

32033

In [ ]:
print(words[0])

emma


In [ ]:
b = {}
for w in words[2]:
  chs = ['<S>'] + list(w) + ['<E>']
  for ch1, ch2 in zip(chs, chs[1:]):
    bigram = (ch1, ch2)
    b[bigram] = b.get(bigram, 0) + 1
    print(bigram, "-", b[bigram])

('<S>', 'a') - 1
('a', '<E>') - 1
('<S>', 'v') - 1
('v', '<E>') - 1
('<S>', 'a') - 2
('a', '<E>') - 2


In [ ]:
for w in words[:2]:
  chs = ['<S>'] + list(w) + ['<E>']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    print(ch1, ch2, ch3)

chs = ['<S>'] + list(w) + ['<E>']
print(chs)

<S> e m
e m m
m m a
m a <E>
<S> o l
o l i
l i v
i v i
v i a
i a <E>
['<S>', 'o', 'l', 'i', 'v', 'i', 'a', '<E>']


In [ ]:
import torch

In [ ]:
N = torch.zeros((27, 27, 27), dtype=torch.int32)

### Character to Integer Mapping

To represent our characters numerically, we'll create a mapping from each character (a-z and a special start/end token `<S>`) to a unique integer. This is essential for working with tensors like `N`.

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0 # Using '.' as a special start/end token, mapping to 0
itos = {i:s for s,i in stoi.items()}

# Let's see the mappings
print("String-to-Integer mapping (stoi):")
print(stoi)
print("\nInteger-to-String mapping (itos):")
print(itos)


String-to-Integer mapping (stoi):
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}

Integer-to-String mapping (itos):
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


### Embedding Function

Now, we'll create a function that takes a list of words, processes each word to generate trigrams, and uses these trigrams to increment counts in our `N` tensor. The `N` tensor will store the frequencies of character sequences, which is a fundamental step in building a character-level language model.

In [ ]:
def embed_words_into_trigrams(word_list, N, stoi):
    for w in word_list:
        # Add start and end tokens
        chs = ['.'] + list(w) + ['.']
        # Iterate through trigrams
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            ix3 = stoi[ch3]
            N[ix1, ix2, ix3] += 1

In [ ]:
tens = (torch.rand((2, 3, 4)) * 10).round().int()

In [ ]:
print(tens)
print(tens[0, 1])
print(tens[0, 1:, 1:3])

tensor([[[ 2, 10,  8,  2],
         [ 9,  0,  4,  7],
         [ 0, 10, 10,  7]],

        [[ 5,  1,  7, 10],
         [ 6,  7,  3,  6],
         [ 7,  0,  6,  5]]], dtype=torch.int32)
tensor([9, 0, 4, 7], dtype=torch.int32)
tensor([[ 0,  4],
        [10, 10]], dtype=torch.int32)


In [ ]:
embed_words_into_trigrams(words, N, stoi)

In [ ]:
# Print a slice of N to see some updated values (e.g., trigrams starting with '.')
print("N tensor slice for trigrams starting with '.':")
print(N[stoi['.'], :, :].sum(axis=1)) # Sum across the third dimension for a clearer view

N tensor slice for trigrams starting with '.':
tensor([   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
        1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
         134,  535,  929])


In [ ]:
# @title 📊 Trigram Distribution Dashboard {display-mode: "form"}
import json
import numpy as np
from google.colab import output
from IPython.display import HTML

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

# Prepare data for the dashboard
# We'll send the counts for the first 2 characters to the frontend
# To keep it performant, we'll provide the data as a nested list
data_matrix = N.tolist()
chars_list = [itos[i] for i in range(27)]

html_code = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f4f6f8; margin: 0; padding: 20px; }
        .dashboard-container { display: flex; flex-direction: column; gap: 20px; max-width: 1200px; margin: auto; }
        .card { background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }
        .header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; }
        .controls { display: flex; gap: 10px; margin-bottom: 20px; }
        select { padding: 8px; border-radius: 4px; border: 1px solid #ddd; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 400px; }
        table { width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 12px; }
        th, td { border: 1px solid #eee; padding: 4px; text-align: center; }
        th { background-color: #f8f9fa; }
        .kpi-container { display: grid; grid-template-columns: repeat(3, 1fr); gap: 20px; }
        .kpi-card { text-align: center; padding: 15px; }
        .kpi-value { font-size: 24px; font-weight: bold; color: #2c3e50; }
        .kpi-label { font-size: 14px; color: #7f8c8d; }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <div class="header">
            <h2>Trigram Transition Frequencies (N Tensor)</h2>
        </div>

        <div class="kpi-container">
            <div class="card kpi-card">
                <div class="kpi-value" id="total-count">0</div>
                <div class="kpi-label">Total Trigrams Embedded</div>
            </div>
            <div class="card kpi-card">
                <div class="kpi-value">27x27x27</div>
                <div class="kpi-label">Tensor Dimensions</div>
            </div>
            <div class="card kpi-card">
                <div class="kpi-value" id="active-trigrams">0</div>
                <div class="kpi-label">Non-Zero Trigram Counts</div>
            </div>
        </div>

        <div class="card">
            <div class="controls">
                <div>
                    <label>First Char (ch1):</label>
                    <select id="ch1-select"></select>
                </div>
                <div>
                    <label>Second Char (ch2):</label>
                    <select id="ch2-select"></select>
                </div>
            </div>
            <p>Distribution of the 3rd character given the prefix above:</p>
            <div class="canvas-wrapper">
                <canvas id="distributionChart"></canvas>
            </div>
        </div>

        <div class="card">
            <h3>Top Transitions Table</h3>
            <div id="table-container"></div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const N_DATA = DATA_PLACEHOLDER;
        const CHARS = CHARS_PLACEHOLDER;
        let chart = null;

        function init() {
            const sel1 = document.getElementById('ch1-select');
            const sel2 = document.getElementById('ch2-select');

            CHARS.forEach((c, i) => {
                sel1.add(new Option(c, i));
                sel2.add(new Option(c, i));
            });

            sel1.value = 0; // Default to '.'
            sel2.value = 1; // Default to 'a'

            sel1.onchange = updateDashboard;
            sel2.onchange = updateDashboard;

            calculateStats();
            updateDashboard();
        }

        function calculateStats() {
            let total = 0;
            let active = 0;
            for(let i=0; i<27; i++) {
                for(let j=0; j<27; j++) {
                    for(let k=0; k<27; k++) {
                        let val = N_DATA[i][j][k];
                        total += val;
                        if(val > 0) active++;
                    }
                }
            }
            document.getElementById('total-count').innerText = total.toLocaleString();
            document.getElementById('active-trigrams').innerText = active.toLocaleString();
        }

        function updateDashboard() {
            const i = document.getElementById('ch1-select').value;
            const j = document.getElementById('ch2-select').value;
            const distribution = N_DATA[i][j];

            renderChart(distribution);
            renderTable(distribution);
        }

        function renderChart(data) {
            const ctx = document.getElementById('distributionChart').getContext('2d');
            if (chart) chart.destroy();

            chart = new Chart(ctx, {
                type: 'bar',
                data: {
                    labels: CHARS,
                    datasets: [{
                        label: 'Frequency',
                        data: data,
                        backgroundColor: 'rgba(54, 162, 235, 0.6)',
                        borderColor: 'rgba(54, 162, 235, 1)',
                        borderWidth: 1
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    scales: { y: { beginAtZero: true } }
                }
            });
        }

        function renderTable(data) {
            let html = '<table><tr><th>Char</th><th>Count</th></tr>';
            let sorted = data.map((v, idx) => ({char: CHARS[idx], val: v}))
                             .sort((a, b) => b.val - a.val)
                             .filter(x => x.val > 0);

            if(sorted.length === 0) {
                html += '<tr><td colspan="2">No data for this sequence</td></tr>';
            } else {
                sorted.forEach(item => {
                    html += `<tr><td>${item.char}</td><td>${item.val}</td></tr>`;
                });
            }
            html += '</table>';
            document.getElementById('table-container').innerHTML = html;
        }

        init();
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json.dumps(data_matrix)).replace('CHARS_PLACEHOLDER', json.dumps(chars_list))

HTML(html_code)

In [ ]:
# @title
import pandas as pd
from IPython.display import display, HTML

# Create list of characters for labels
labels = [itos[i] for i in range(27)]

# Helper to create a pretty table for a slice of N
def display_n_slice(index, title):
    df = pd.DataFrame(N[index].numpy(), index=labels, columns=labels)
    # Style the dataframe to highlight non-zero values
    styled_df = df.style.background_gradient(cmap='Blues').set_caption(title)
    display(HTML(f"<h3>{title}</h3>"))
    display(styled_df)

# Display the first element (N[0]) - Trigrams starting with '.'
display_n_slice(0, "N[0]: Trigrams starting with '.' (First character is start token)")

# Display the second element (N[1]) - Trigrams starting with 'a'
display_n_slice(1, "N[1]: Trigrams starting with 'a'")

,.,a,b,c,d,e,f,g,h,i,j,k,l,m,n,o,p,q,r,s,t,u,v,w,x,y,z
.,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
a,0,207,190,31,366,55,21,17,91,154,27,75,632,384,623,10,17,9,482,194,72,152,243,6,27,173,152
b,0,169,0,0,0,253,0,0,9,41,1,0,85,0,0,77,0,0,646,0,0,21,0,0,0,4,0
c,0,628,0,0,0,65,0,0,352,44,2,0,68,0,0,255,0,0,67,0,0,13,0,0,0,46,2
d,0,700,0,0,0,524,0,0,32,130,6,2,2,19,3,119,0,0,77,1,0,35,3,6,0,31,0
e,0,23,15,4,80,9,10,6,9,54,3,8,488,288,60,4,8,0,90,93,20,21,154,4,4,24,52
f,0,158,0,0,0,49,1,0,0,71,0,0,20,0,0,25,0,0,79,0,0,6,0,0,0,8,0
g,0,136,0,0,0,110,0,0,10,128,0,0,18,0,0,27,0,0,165,0,0,45,0,25,0,5,0
h,0,505,0,0,0,151,0,0,0,55,0,0,0,0,0,77,0,0,6,2,0,67,0,0,0,11,0
i,0,9,12,6,21,1,10,4,5,1,0,17,67,37,60,9,1,2,49,124,24,0,49,2,2,20,59


,.,a,b,c,d,e,f,g,h,i,j,k,l,m,n,o,p,q,r,s,t,u,v,w,x,y,z
.,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
a,40,0,5,5,47,0,3,3,17,12,2,12,46,23,174,0,0,3,83,28,8,0,14,0,0,20,11
b,36,28,20,1,65,173,0,0,18,76,0,0,10,0,2,4,0,0,85,5,0,6,0,0,0,12,0
c,11,38,0,8,0,129,0,1,77,56,0,59,26,0,0,24,1,11,0,3,0,1,0,0,2,23,0
d,106,164,1,0,94,194,2,1,36,190,2,1,33,3,4,59,0,1,69,2,0,5,12,5,0,57,1
e,147,9,7,5,53,8,1,17,13,1,3,2,287,10,13,3,0,2,29,32,11,2,28,0,0,6,3
f,14,29,0,0,0,13,10,0,0,36,0,0,0,0,3,3,0,0,6,6,10,0,0,4,0,0,0
g,2,42,0,0,12,26,0,6,15,7,0,0,2,1,11,17,0,0,16,0,0,7,0,0,0,4,0
h,1714,111,3,2,11,46,1,0,1,101,7,15,73,88,57,9,1,1,34,17,1,8,5,1,0,11,14
i,183,105,14,16,138,15,7,20,5,10,14,26,259,51,177,14,1,3,189,145,50,4,32,3,1,101,67


In [ ]:
P = N.float() + 1

In [ ]:
torch.randn(1).item()

0.9465835094451904

In [ ]:
P

tensor([[[  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [  1., 208., 191.,  ...,  28., 174., 153.],
         [  1., 170.,   1.,  ...,   1.,   5.,   1.],
         ...,
         [  1.,  58.,   1.,  ...,   2.,  18.,  12.],
         [  1., 247.,   1.,  ...,   1.,   1.,   3.],
         [  1., 457.,   1.,  ...,   1.,  92.,   2.]],

        [[  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [ 41.,   1.,   6.,  ...,   1.,  21.,  12.],
         [ 37.,  29.,  21.,  ...,   1.,  13.,   1.],
         ...,
         [ 12.,   6.,   1.,  ...,  18.,   7.,   4.],
         [164., 390.,  14.,  ...,   1.,  17.,  41.],
         [ 39., 124.,   1.,  ...,   1.,  13.,  23.]],

        [[  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [ 47.,   6.,   6.,  ...,   5.,  32.,   5.],
         [  2.,   9.,   1.,  ...,   1.,  10.,   1.],
         ...,
         [  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [ 56.,   5.,   2.,  ...,   1.,   1.,   1.],
         [  1.,   1.,   1.,  ...,   1.,   1.,   1.]],

In [ ]:
ix = torch.multinomial(P[0, 1], num_samples=1, replacement=True).item()
itos[ix]

'i'

In [ ]:
import random
for i in range(100):
  out = []
  starting = 0
  ix = random.randint(1, 26)
  while True:
    p = P[starting, ix] / P[starting, ix].sum()

    new_word = torch.multinomial(p, num_samples=1, replacement=True).item()
    ix = new_word
    starting = ix
    out.append(itos[new_word])
    if ix == 0:
      break
  print(''.join(out))

are.
an.
al.
i.
asdi.
vyyvot.
an.
ua.
riqwpsi.
lyi.
odi.
aycmi.
okwkoseydhwhfubi.
rodarkixsa.
an.
xlori.
ie.
vsanemayurenariuzasi.
enemade.
ri.
oqkorarac.
iu.
apnane.
iqsasombwqmyakodali.
larasenokamk.
ane.
emabaruwqvodrezgdivyadonadadaryhkadi.
abhfp.
an.
ulele.
anfi.
onalechle.
uubvolybfsdi.
upn.
azanygiriryaypalanyan.
an.
arin.
irib.
i.
ry.
elygydyari.
dariayanalaslyale.
zad.
hyora.
zarix.
ani.
orymi.
i.
renii.
l.
ode.
afi.
naranelyi.
ryan.
yayaya.
ri.
mazgbhgi.
edtoe.
avanemakan.
ealyl.
anori.
i.
kren.
i.
i.
yte.
arasiwwjcari.
magzupcvaneri.
anenaryi.
i.
aduonodibfi.
asitafrimalinamehsazlat.
ooryan.
an.
di.
wzkiarokri.
asi.
evi.
enealaro.
ahosayy.
e.
byahicri.
asanosavzuucjmevkari.
upododokvyzagsareanl.
relyqpsysayqvi.
alindaun.
dme.
rhzyzanalone.
ayosi.
i.
i.
bi.
vx.
iriuyal.
ialene.
lemarickpyi.
ehanat.
adie.
ananeukhkemiarinadi.
lanan.


## Using NN approach!

In [ ]:
# Craeting the training dataset
xs, ys, zs = [], [], []

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    xs.append(ix1)
    ys.append(ix2)
    zs.append(ix3)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
zs = torch.tensor(zs)
n = xs.nelement()
print(f"Number of examples: {n}")

Number of examples: 196113


In [ ]:
# Initialize the 'network'
w = torch.rand((27, 27), requires_grad=True)

In [ ]:
loss = -probs[torch.arange(n), ys].log().mean()
loss

tensor(2.6481, grad_fn=<NegBackward0>)

In [ ]:
probs.shape

torch.Size([196113, 27])

In [ ]:
import torch.nn.functional as F

# Gradient Descent
for i in range(1):
  xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one hot encoding
  yenc = F.one_hot(ys, num_classes=27).float()
  inputs = torch.tensor(xenc, yenc).sum(1)
  logits = (inputs @ w) # predict log counts
  counts = logits.exp() # counts, equivalant to N matrix
  probs = counts / counts.sum(1, keepdim=True) # probabilities for the next character
  loss = -probs[torch.arange(n), ys].log().mean()

  # backward pass
  w.grad = None # set to zero the gradient
  loss.backward()

  # update
  w.data += -4 * w.grad

print(loss.item())

TypeError: tensor() takes 1 positional argument but 2 were given